# ZOD04 — One ranked rule per label plus fallback

This notebook uses the first eligible trust-v005 rule for each of the 18 labels, then predicts remaining rows with the V1-all → V0 UltraBERT fallback (N0089). On the frozen V0 development set this produced 342/540 (63.33%); this is not an independent generalization estimate. It caches each selected checkpoint's top-1 predictions once and writes `submission.parquet` with exactly `id,model`. Internet is disabled; model bases are attached as Kaggle sources.


In [ ]:
from pathlib import Path
import gc, json, os, subprocess, sys, time
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
INPUT_ROOT = Path("/kaggle/input")
asset_manifests = list(INPUT_ROOT.rglob("zod04_manifest.json"))
assert len(asset_manifests) == 1, f"expected one ZOD04 asset manifest; found {asset_manifests}"
DATA = asset_manifests[0].parent
BUNDLE = json.loads((DATA / "zod04_manifest.json").read_text())
assert BUNDLE["trust_list_version"] == "trust-v005"
assert BUNDLE["algorithm"] == "first_eligible_precision_ranked_rule_per_label_then_fallback"
assert len(BUNDLE["rules"]) == 18 and len(BUNDLE["candidates"]) == 16
from importlib.metadata import version
SKLEARN_VERSION = "1.9.1"
if version("scikit-learn") != SKLEARN_VERSION:
    wheel = next(INPUT_ROOT.rglob("scikit_learn-1.9.1-*.whl"), None)
    assert wheel is not None, "pinned scikit-learn wheel missing"
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", "--force-reinstall", str(wheel)], check=True)
import joblib, numpy as np, pandas as pd, torch
assert torch.cuda.is_available(), "Kaggle GPU unavailable"
WORK = Path("/kaggle/working/zod04_runtime")
SRC = WORK / "src"
SRC.mkdir(parents=True, exist_ok=True)
for name in ("__init__.py", "data_factory.py", "feature_factory.py", "model_factory.py", "inference.py"):
    (SRC / name).symlink_to(DATA / f"source__{name}")
sys.path.insert(0, str(WORK))
CHECKPOINTS = WORK / "checkpoints"
CHECKPOINTS.mkdir(parents=True, exist_ok=True)
zod03_manifests = list(INPUT_ROOT.rglob("zod03_manifest.json"))
assert len(zod03_manifests) == 1, f"expected one ZOD03 base manifest; found {zod03_manifests}"
ZOD03_DATA = zod03_manifests[0].parent
ZOD03 = json.loads(zod03_manifests[0].read_text())
for sequence_id, candidate in BUNDLE["candidates"].items():
    dest = CHECKPOINTS / sequence_id
    dest.mkdir(parents=True, exist_ok=True)
    source_root = ZOD03_DATA if candidate["asset_source"] == "zod03-models" else DATA
    for item in candidate["files"]:
        source = source_root / item["flat_filename"]
        assert source.is_file(), f"checkpoint asset missing: {sequence_id}/{item['flat_filename']}"
        target = dest / item["relative_path"]
        target.parent.mkdir(parents=True, exist_ok=True)
        target.symlink_to(source)
    candidate["runtime_checkpoint"] = str(dest if candidate["checkpoint_type"] == "directory" else dest / candidate["files"][0]["relative_path"])

BASES = {}
def resolve_model(marker):
    matches = [p.parent for p in INPUT_ROOT.rglob("config.json") if marker.lower() in str(p.parent).lower()]
    assert matches, f"no attached model found for {marker}"
    # Some sentence-transformer sources contain nested pooling/normalization
    # configs. The model root is the shallowest matching config directory.
    depth = min(len(path.parts) for path in matches)
    roots = sorted({path for path in matches if len(path.parts) == depth})
    assert len(roots) == 1, f"expected one root model for {marker}; found {roots}"
    return roots[0]
for model, marker in (("gemma4_e2b", "gemma-4-e2b"), ("embeddinggemma2", "embeddinggemma-2"),
                      ("ministral3_3b", "ministral-3-3b-base-2512"), ("qwen06", "qwen3-0-6b")):
    if model in {c["model"] for c in BUNDLE["candidates"].values()}:
        BASES[model] = resolve_model(marker)
for model in ("granite4_350m", "granite4_h_micro"):
    candidate = next(c for c in BUNDLE["candidates"].values() if c["model"] == model)
    ref = next((c for c in ZOD03["candidates"].values()
                if c.get("base_model") == candidate["base_model"]
                and c.get("base_model_revision") == candidate["base_model_revision"]
                and c.get("base_files")), None)
    assert ref is not None, f"exact Granite base not found in ZOD03 assets: {model}"
    dest = WORK / "bases" / model
    dest.mkdir(parents=True, exist_ok=True)
    for item in ref["base_files"]:
        source = ZOD03_DATA / item["flat_filename"]
        target = dest / item["relative_path"]
        target.parent.mkdir(parents=True, exist_ok=True)
        target.symlink_to(source)
    BASES[model] = dest
llama_manifests = list(INPUT_ROOT.rglob("llama_base_manifest.json"))
assert len(llama_manifests) == 1, f"expected one Llama base manifest; found {llama_manifests}"
if "llama32_3b" in {c["model"] for c in BUNDLE["candidates"].values()}:
    info = json.loads(llama_manifests[0].read_text())
    candidate = next(c for c in BUNDLE["candidates"].values() if c["model"] == "llama32_3b")
    assert info["model"] == candidate["base_model"] and info["revision"] == candidate["base_model_revision"]
    dest = WORK / "bases" / "llama32_3b"
    dest.mkdir(parents=True, exist_ok=True)
    for item in info["files"]:
        source = llama_manifests[0].parent / item["flat_filename"]
        target = dest / item["relative_path"]
        target.parent.mkdir(parents=True, exist_ok=True)
        target.symlink_to(source)
    BASES["llama32_3b"] = dest
try:
    from importlib.metadata import PackageNotFoundError
    from packaging.version import Version
    if Version(version("torchao")) <= Version("0.16.0"):
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True, capture_output=True)
except (PackageNotFoundError, ImportError):
    pass
from src.inference import _predict_neural, predict_catboost_stylometry, predict_svc_bundle
TEST = next((p for p in (INPUT_ROOT / "competitions/name-that-llm/test.csv", INPUT_ROOT / "name-that-llm/test.csv") if p.is_file()), None)
assert TEST is not None, "competition test.csv missing"
print({"algorithm": BUNDLE["algorithm"], "trust_version": BUNDLE["trust_list_version"],
       "rules": len(BUNDLE["rules"]), "checkpoints": len(BUNDLE["candidates"]),
       "fallback": BUNDLE["fallback_sequence_id"]})

In [ ]:
test = pd.read_csv(TEST, dtype={"id": "string"})
assert {"id", "text"}.issubset(test.columns)
assert test.id.notna().all() and not test.id.duplicated().any()
test.text = test.text.fillna("").astype(str)
classes = list(map(str, BUNDLE["classes"]))
texts, n = test.text.tolist(), len(test)
def get_top1(sequence_id):
    candidate = BUNDLE["candidates"][sequence_id]
    model, checkpoint = candidate["model"], Path(candidate["runtime_checkpoint"])
    if model == "svc":
        probabilities, labels = predict_svc_bundle(joblib.load(checkpoint), texts)
    elif model == "catboost_stylometry":
        probabilities, labels = predict_catboost_stylometry(checkpoint, texts, classes)
    else:
        base = BASES.get(model)
        if model != "ultrabert380" and candidate.get("base_model"):
            assert base is not None, f"base model was not resolved for {model}"
        batch_size = min(4, max(1, int(candidate["config"].get("batch_size", 4) or 4)))
        while True:
            try:
                probabilities, labels = _predict_neural(
                    model, checkpoint, texts, classes, batch_size=batch_size,
                    max_length=int(candidate["config"].get("max_length", 384)),
                    base_checkpoint=str(base) if base else None, revision=None,
                    text_only_backbone=bool(candidate["config"].get("text_only_backbone", False)))
                break
            except RuntimeError as exc:
                if "out of memory" not in str(exc).lower() or batch_size <= 1:
                    raise
                batch_size = max(1, batch_size // 2)
                gc.collect(); torch.cuda.empty_cache()
    labels = list(map(str, labels))
    class_order = [labels.index(label) for label in classes]
    prediction = np.asarray(classes, dtype=object)[np.asarray(probabilities)[:, class_order].argmax(axis=1)]
    del probabilities; gc.collect(); torch.cuda.empty_cache()
    return prediction

started = time.time()
checkpoint_order = list(dict.fromkeys([r["sequence_id"] for r in BUNDLE["rules"]] + [BUNDLE["fallback_sequence_id"]]))
assert set(checkpoint_order) == set(BUNDLE["candidates"])
top1_cache = {sequence_id: get_top1(sequence_id) for sequence_id in checkpoint_order}
pending = np.ones(n, dtype=bool)
output = np.empty(n, dtype=object)
source = np.empty(n, dtype=object)
trace = []
for rule in BUNDLE["rules"]:
    sequence_id, label = rule["sequence_id"], rule["label"]
    selected = pending & (top1_cache[sequence_id] == label)
    output[selected], source[selected], pending[selected] = label, sequence_id, False
    trace.append({"order": rule["order"], "trust_sequence": rule["trust_sequence"],
        "sequence_id": sequence_id, "label": label,
        "precision_v0": rule["precision_v0"], "assigned": int(selected.sum()),
        "remaining": int(pending.sum())})
remaining = np.flatnonzero(pending)
fallback = BUNDLE["fallback_sequence_id"]
output[remaining], source[remaining] = top1_cache[fallback][remaining], fallback
submission = pd.DataFrame({"id": test.id, "model": output})
assert submission.columns.tolist() == ["id", "model"] and len(submission) == len(test)
assert submission.id.astype(str).tolist() == test.id.astype(str).tolist()
assert not submission.id.duplicated().any() and submission.id.notna().all()
assert submission.model.notna().all() and submission.model.isin(classes).all()
submission.to_parquet("/kaggle/working/submission.parquet", index=False)
pd.DataFrame(trace).to_csv("/kaggle/working/zod04_rule_counts.csv", index=False)
pd.DataFrame({"id": test.id, "model": submission.model,
              "selected_sequence_id": source}).to_parquet("/kaggle/working/zod04_decisions.parquet", index=False)
print({"approach": "zod04 one eligible rule per label plus fallback", "rows": len(submission),
       "runtime_seconds": round(time.time() - started, 1), "schema": submission.columns.tolist(),
       "cached_checkpoints": len(top1_cache), "output": "/kaggle/working/submission.parquet"})
display(pd.DataFrame(trace)); display(submission.head())